In [ ]:
# !pip install simple-icd-10 simple-icd-10-cm --quiet

In [1]:
import os
import re
import numpy as np
import pandas as pd
from IPython.display import display
import simple_icd_10 as icd
import simple_icd_10_cm as icd_cm
from functools import partial

In [2]:
df = pd.read_csv("trajectories.csv")
df.shape

(5938215, 11)

In [3]:
df.head()

,subject_id,hadm_id,seq_num,icd10_code,icd10_category,admittime,dischtime,deathtime,gender,birth_year,dod
0,10000032,22595853,1,K766,K76,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,F,2128,2180-09-09
1,10000032,22595853,2,R188,R18,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,F,2128,2180-09-09
2,10000032,22595853,3,K740,K74,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,F,2128,2180-09-09
3,10000032,22595853,4,B1920,B19,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,F,2128,2180-09-09
4,10000032,22595853,5,J449,J44,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,F,2128,2180-09-09


# Getting descriptions of ICD10 codes

In [4]:
categories = df.groupby(by="icd10_category").agg({"icd10_code": "count"}).reset_index().rename(columns={"icd10_code": "count"})

In [5]:
categories.head()

,icd10_category,count
0,A01,13
1,A02,137
2,A03,36
3,A04,7034
4,A05,78


In [6]:
def get_description(icd10_category, lib=icd):
    try:
        return lib.get_description(icd10_category)
    except:
        return None

In [7]:
categories["description"] = categories["icd10_category"].apply(partial(get_description, lib=icd))

In [8]:
categories.head()

,icd10_category,count,description
0,A01,13,Typhoid and paratyphoid fevers
1,A02,137,Other salmonella infections
2,A03,36,Shigellosis
3,A04,7034,Other bacterial intestinal infections
4,A05,78,"Other bacterial foodborne intoxications, not e..."


In [9]:
categories.count()

icd10_category    1757
count             1757
description       1696
dtype: int64

In [10]:
categories[categories["description"].isna()]

,icd10_category,count,description
66,A90,13,None
67,A91,1,None
81,B10,76,None
117,B59,247,None
185,C4A,46,None
...,...,...,...
1725,Z68,56368,None
1726,Z69,3,None
1734,Z77,995,None
1735,Z78,10190,None


In [11]:
categories.loc[categories["description"].isna(), "description"] = categories.loc[categories["description"].isna(), "icd10_category"].apply(partial(get_description, lib=icd_cm))

In [12]:
categories.count()

icd10_category    1757
count             1757
description       1756
dtype: int64

In [13]:
categories[categories['description'].isna()]

,icd10_category,count,description
1025,NOD,30585,None


In [14]:
categories.loc[categories["icd10_category"].isin(["NOD"]), "description"] = "Unknown code"

In [15]:
categories.count()

icd10_category    1757
count             1757
description       1757
dtype: int64

In [16]:
categories.head()

,icd10_category,count,description
0,A01,13,Typhoid and paratyphoid fevers
1,A02,137,Other salmonella infections
2,A03,36,Shigellosis
3,A04,7034,Other bacterial intestinal infections
4,A05,78,"Other bacterial foodborne intoxications, not e..."


In [17]:
categories.to_csv("categories.tsv", sep="\t", index=False)

# Infer embeddings of descriptions

In [1]:
# !pip uninstall -y tensorflow tensorflow-gpu tensorflow-cpu tensorflow-metadata keras keras-preprocessing
# !pip install -U "transformers" "tokenizers" "sentencepiece" "protobuf"

In [2]:
# !pip check

In [1]:
import json
import pandas as pd
import numpy as np
import torch
from huggingface_hub import hf_hub_download
from safetensors import safe_open
from transformers import AutoTokenizer
from typing import List, Optional
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
from pickle import dump, load

from openai_harmony import (
    Author,
    Conversation,
    DeveloperContent,
    HarmonyEncodingName,
    Message,
    Role,
    SystemContent,
    ToolDescription,
    load_harmony_encoding,
    ReasoningEffort
)
from enum import Enum
from typing import Iterable, Optional, List, Dict

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODELS = [
    # "Qwen/Qwen3-235B-A22B-Instruct-2507",
    # "yandex/YandexGPT-5-Lite-8B-instruct",
    # "deepseek-ai/DeepSeek-V3",
    # "mistralai/Mistral-Small-3.2-24B-Instruct-2506",
    # "openai/gpt-oss-20b",
    "openai/gpt-oss-120b"
]

/home/d.kornilov/work/Revealing-interconnections-between-diseases/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
categories = pd.read_csv("datasets/categories.tsv", sep="\t")

In [3]:
categories.head()

,icd10_category,count,description
0,A01,13,Typhoid and paratyphoid fevers
1,A02,137,Other salmonella infections
2,A03,36,Shigellosis
3,A04,7034,Other bacterial intestinal infections
4,A05,78,"Other bacterial foodborne intoxications, not e..."


In [2]:
class OpenAITokenizer():
    def __init__(
        self,
        reasoning_effort: ReasoningEffort = ReasoningEffort.LOW,
        conversation_start_date: str = "2025-06-28",
        developer_message: str = "",
    ):
        self.set_system_message(reasoning_effort=reasoning_effort, conversation_start_date=conversation_start_date)
        self.set_developer_message(message=developer_message)
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)

    def set_system_message(self, reasoning_effort: ReasoningEffort, conversation_start_date: str):
        self.system_message = (
            SystemContent.new()
                .with_reasoning_effort(reasoning_effort)
                .with_conversation_start_date(conversation_start_date)
        )

    def set_developer_message(self, message: str = ""):
        self.developer_message = (
            DeveloperContent.new()
            .with_instructions(message)
        )

    def __call__(
            self, 
            query: str, 
            return_tensors: str="pt" # for compatibility, not used
        ) -> Dict[str, torch.Tensor]:
        convo = Conversation.from_messages(
            [
                Message.from_role_and_content(Role.SYSTEM, self.system_message),
                Message.from_role_and_content(Role.DEVELOPER, self.developer_message),
                Message.from_role_and_content(Role.USER, query),
            ]
        )
        tokens = self.encoding.render_conversation_for_completion(convo, Role.ASSISTANT)
        input_ids = torch.tensor(tokens, dtype=torch.long)
        return {"input_ids": input_ids}
    
def load_tok(
        model_id: str, 
        reasoning_effort: ReasoningEffort = ReasoningEffort.LOW, 
        conversation_start_date: str = "2025-06-28", 
        developer_message: str = ""
    ):
    if model_id.startswith("openai/"):
        return OpenAITokenizer(reasoning_effort=reasoning_effort, conversation_start_date=conversation_start_date, developer_message=developer_message)
    else:
        try:
            return AutoTokenizer.from_pretrained(model_id, use_fast=True)
        except Exception:
            # slow SPM path (requires sentencepiece and protobuf compatible with it)
            return AutoTokenizer.from_pretrained(model_id, use_fast=False)

def get_embedding_weight(model_id: str, device: str="cpu"):
    # 1) Find which shard has the embedding weight
    print("Downloading index...")
    index_path = hf_hub_download(model_id, filename="model.safetensors.index.json")
    with open(index_path, "r") as f:
        index = json.load(f)
    weight_key = "model.embed_tokens.weight"
    shard_file = index["weight_map"][weight_key]
    
    # 2) Download only that shard and read the single tensor
    print("Downloading weights...")
    shard_path = hf_hub_download(model_id, filename=shard_file)
    with safe_open(shard_path, framework="pt", device=device) as f:
        W = f.get_tensor(weight_key)                  # (vocab_size, hidden_dim)
    return W

def infer_embedding_layer(icd10_category, description, W, tok, device="cpu"):
    ids = tok(description, return_tensors="pt")["input_ids"]  # CPU is fine; move to GPU if you want
    emb = torch.nn.functional.embedding(ids.to(device), W.to(device)).to("cpu")
    return icd10_category, emb

def get_embeddings(
        model_id: str, 
        descriptions: pd.DataFrame,
        device: str="cpu", 
        max_descriptions: Optional[int]=None,
        reasoning_effort: ReasoningEffort = ReasoningEffort.LOW, 
        conversation_start_date: str = "2025-06-28", 
        developer_message: str = ""
    ):
    print("Processing with", model_id, "...")
    tok = load_tok(model_id, reasoning_effort=reasoning_effort, conversation_start_date=conversation_start_date, developer_message=developer_message)
    weight = get_embedding_weight(model_id, device=device)
    embs = {}
    for idx, row in tqdm(
        descriptions[:max_descriptions].iterrows(), 
        total=len(descriptions[:max_descriptions]), 
        desc="Computing embeddings..."
    ):
        icd10_category, emb = infer_embedding_layer(row["icd10_category"], row["description"], weight, tok, device=device)
        embs[icd10_category] = emb

    return embs

In [5]:
for model_id in MODELS:
    for reasoning_effort in tqdm([ReasoningEffort.LOW, ReasoningEffort.MEDIUM, ReasoningEffort.HIGH]):
        embs = get_embeddings(
            model_id=model_id, 
            descriptions=categories, 
            # max_descriptions=5,
            reasoning_effort=reasoning_effort,
        )
        with open("embeddings/"+f"{model_id}_{reasoning_effort}.pkl".replace("/", "_"), "wb") as f:
            dump(embs, f)

  0%|          | 0/3 [00:00<?, ?it/s]

Processing with openai/gpt-oss-120b ...


 33%|███▎      | 1/3 [01:38<03:17, 98.60s/it]

Processing with openai/gpt-oss-120b ...


 67%|██████▋   | 2/3 [03:16<01:37, 98.00s/it]

Processing with openai/gpt-oss-120b ...


100%|██████████| 3/3 [04:53<00:00, 97.82s/it]


In [4]:
embs = {}

for model_id in MODELS:
    for reasoning_effort in [ReasoningEffort.LOW, ReasoningEffort.MEDIUM, ReasoningEffort.HIGH]:
        print(model_id, reasoning_effort)
        with open("embeddings/"+f"{model_id}_{reasoning_effort}.pkl".replace("/", "_"), "rb") as f: embs = load(f)
        print(embs['A01'].shape)

        for icd10_category in tqdm(embs):
            # embs[icd10_category] = embs[icd10_category].squeeze().max(axis=0).values #mean/max
            embs[icd10_category] = embs[icd10_category].squeeze().mean(axis=0)
        print(embs['A01'].shape)

        # with open("embeddings/"+f"{model_id}_{reasoning_effort}_max.pkl".replace("/", "_"), "wb") as f:
        with open("embeddings/"+f"{model_id}_{reasoning_effort}_mean.pkl".replace("/", "_"), "wb") as f:
            dump(embs, f)

openai/gpt-oss-120b ReasoningEffort.LOW
torch.Size([84, 2880])


100%|██████████| 1757/1757 [03:04<00:00,  9.55it/s]


torch.Size([2880])
openai/gpt-oss-120b ReasoningEffort.MEDIUM
torch.Size([84, 2880])


100%|██████████| 1757/1757 [03:05<00:00,  9.46it/s]


torch.Size([2880])
openai/gpt-oss-120b ReasoningEffort.HIGH
torch.Size([84, 2880])


100%|██████████| 1757/1757 [03:05<00:00,  9.50it/s]


torch.Size([2880])
